In [ ]:
!gdown 14lX_JgofYZLbIkjPf7rXmBOpVIfMudDo -O kaggle.json
!mkdir -p /root/.kaggle
!cp kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

!kaggle datasets download thedevastator/dailydialog-multi-turn-dialog-with-intention-and
!unzip dailydialog-multi-turn-dialog-with-intention-and.zip -d .

Downloading...
From: https://drive.google.com/uc?id=14lX_JgofYZLbIkjPf7rXmBOpVIfMudDo
To: /content/kaggle.json
100% 69.0/69.0 [00:00<00:00, 314kB/s]
Dataset URL: https://www.kaggle.com/datasets/thedevastator/dailydialog-multi-turn-dialog-with-intention-and
License(s): CC0-1.0
Archive:  dailydialog-multi-turn-dialog-with-intention-and.zip
  inflating: ./test.csv              
  inflating: ./train.csv             
  inflating: ./validation.csv        


In [ ]:
import pandas as pd
import ast
import re

# Function to fix act and emotion columns (adds missing commas and converts to list)
def fix_and_eval_list(s):
    if isinstance(s, list):
        return s
    s_fixed = re.sub(r"(\d)\s+(?=\d)", r"\1, ", s)
    return ast.literal_eval(s_fixed)

# Function to parse dialog and extract utterances (between quotes)
def parse_dialog(s):
    utterances = re.findall(r"'(.*?)'|\"(.*?)\"", s)
    return [u[0] if u[0] else u[1] for u in utterances]

def load_and_prepare_flat_df(path):
    # Load CSV
    df = pd.read_csv(path)

    # Parse columns
    df["dialog"] = df["dialog"].apply(parse_dialog)
    df["act"] = df["act"].apply(fix_and_eval_list)
    df["emotion"] = df["emotion"].apply(fix_and_eval_list)

    # Flatten
    rows = []
    for dialog_id, row in df.iterrows():
        for turn_id, (utt, emo) in enumerate(zip(row["dialog"], row["emotion"])):
            rows.append({
                "dialog_id": dialog_id,
                "turn_id": turn_id,
                "utterance": utt,
                "label": emo
            })
    df_flat = pd.DataFrame(rows)



    return df_flat



In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-mpnet-base-v2')  # o 'all-MiniLM-L6-v2'

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.4k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'max_seq_length': 384, 'do_lower_case': False}) with Transformer model: MPNetModel 
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [ ]:
df_train_flat = load_and_prepare_flat_df("train.csv")
df_val_flat = load_and_prepare_flat_df("validation.csv")
df_test_flat = load_and_prepare_flat_df("test.csv")

In [ ]:
df_train_flat.head()

,dialog_id,turn_id,utterance,label
0,0,0,"Say , Jim , how about going for a few beers af...",0
1,0,1,You know that is tempting but is really not g...,0
2,0,2,What do you mean ? It will help us to relax .,0
3,0,3,Do you really think so ? I don't . It will ju...,0
4,0,4,I guess you are right.But what shall we do ? ...,0


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np


# Utterances e label
utterances = df_train_flat["utterance"].tolist()
labels = df_train_flat["label"].values

# Encoding diretto con batching interno
embeddings = model.encode(
    utterances,
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True
)

# Salva su disco
np.save("train_embeddings.npy", embeddings)
np.save("train_labels.npy", labels)


Batches:   0%|          | 0/341 [00:00<?, ?it/s]

In [ ]:
# Utterances e label per il validation set
val_utterances = df_val_flat["utterance"].tolist()
val_labels = df_val_flat["label"].values

# Encoding
val_embeddings = model.encode(
    val_utterances,
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True
)

# Salva su disco
np.save("val_embeddings.npy", val_embeddings)
np.save("val_labels.npy", val_labels)


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [ ]:
# Utterances e label per il test set
test_utterances = df_test_flat["utterance"].tolist()
test_labels = df_test_flat["label"].values

# Encoding
test_embeddings = model.encode(
    test_utterances,
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True
)

# Salva su disco
np.save("test_embeddings.npy", test_embeddings)
np.save("test_labels.npy", test_labels)


Batches:   0%|          | 0/31 [00:00<?, ?it/s]

In [ ]:
import zipfile

# Train set
with zipfile.ZipFile('train_embeddings_labels_sentence.zip', 'w') as zipf:
    zipf.write('train_embeddings.npy')
    zipf.write('train_labels.npy')

# Validation set
with zipfile.ZipFile('val_embeddings_labels_sentence.zip', 'w') as zipf:
    zipf.write('val_embeddings.npy')
    zipf.write('val_labels.npy')

# Test set
with zipfile.ZipFile('test_embeddings_labels_sentence.zip', 'w') as zipf:
    zipf.write('test_embeddings.npy')
    zipf.write('test_labels.npy')


In [ ]:
embeddings.shape

(87170, 768)